# Dealing with Conflicts

This file shows how `Menhir` deals with *shift-reduce* and *reduce-reduce* conflicts.  (The Python version of this
notebook uses `Lark`.)
The following grammar is *ambiguous* because it does not specify the precedence of the arithmetical operators:
```
    expr : expr "+" expr
         | expr "*" expr
         | NUMBER
```

## Specification of the Scanner and the Parser

The scanner specification `lexer` for `ocamllex` defines the tokens, and the grammar `grammar` for `Menhir` defines
the structure.  In the `Lark` grammar of the Python version, the operators `"+"` and `"*"` occur as *anonymous
terminals* and `Lark` invents the names `PLUS` and `STAR` for them.  `Menhir` needs every token to be declared, so we
declare the tokens `PLUS` and `STAR` ourselves.  These names will show up in the messages discussed below.

`Menhir` needs a start symbol that is followed by the end of the input.  Therefore, there is an additional rule
`start : expr EOF`.  It corresponds to the rule `$root_expr : expr` that `Lark` adds to its grammars.

The semantic actions create nodes of a parse tree in the style of `Lark`:  the names `add`, `mul`, and `number`
correspond to the *aliases* `-> add`, `-> mul`, and `-> number` of the `Lark` grammar.  We will use these names
later in order to build an abstract syntax tree.  The semantic actions only change the result of a rule, they have no
influence on the parse table.

In [ ]:
let lexer = {|
{
  open Expr_parser
}

rule token = parse
  | [' ' '\t' '\r' '\n']+        { token lexbuf }      (* %ignore WS *)
  | '+'                          { PLUS }
  | '*'                          { STAR }
  | '0' | ['1'-'9'] ['0'-'9']*   { NUMBER (Parse_tree.token "NUMBER" lexbuf) }
  | eof                          { EOF }
  | _                            { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ repr (Lexing.lexeme lexbuf))) }
|}

let grammar = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> NUMBER
%token PLUS STAR EOF

%start <Parse_tree.tree> start
%type  <Parse_tree.tree> expr

%%

start: e = expr EOF                { e }

expr : l = expr PLUS r = expr      { Tree ("add", [l; r]) }
     | l = expr STAR r = expr      { Tree ("mul", [l; r]) }
     | n = NUMBER                  { Tree ("number", [Token n]) }
|}

## Making `Menhir` Talk About Its Conflicts

`Lark` is quite tight-lipped about shift-reduce conflicts: by default, the parser is built, the conflicts are resolved
silently, and nothing is printed.  The Python version therefore has to lower the level of the logger `lark.logger`
and create the parser with `debug=True`.

`Menhir` is more talkative:  it always prints a *warning* that tells us how many states have conflicts and how many
conflicts have been resolved.  The option `--explain`, which `generate_parser` always passes to `Menhir`, makes
`Menhir` write a detailed explanation of every conflict into the file `_build/expr_parser.conflicts`.  Hence there is
nothing to configure.  We only load the file `../lib/parser_generator.ml`, which provides the function
`generate_parser` and the module `Parse_tree`.

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
open Parse_tree

`generate_parser` calls `menhir --lalr`, which creates an *LALR(1)* parser.  This corresponds to the keyword argument
`parser='lalr'` of `Lark`.  The *default* parser of `Lark` is an *Earley* parser, and an Earley parser has no problem at
all with an ambiguous grammar: it would simply return one of the possible parse trees without complaining.

In [ ]:
generate_parser ~name:"expr" ~lexer ~grammar

These messages tell us that

1. two states of the LALR automaton have shift-reduce conflicts, and
2. four entries of the action table were ambiguous.

Note the word *arbitrarily*:  if no precedence has been declared, `Menhir` resolves **every** shift-reduce conflict in
favour of shifting, exactly like `Lark` does.  `Menhir` also supports *precedence declarations*, which resolve the
conflicts in a controlled way.  We do not use them here, since the Python version cannot use them either.

The file `_build/expr_parser.conflicts` names, for every conflict,

1. the state and the terminals for which the entry of the action table is ambiguous, and
2. two derivations that show why both shifting and reducing are possible in this situation.

In [ ]:
cat "_build/expr_parser.conflicts"

The only thing we can do is to declare that we do not want to tolerate any conflicts at all.  If `Menhir` is called with
the option `--strict`, then the warnings become errors and no parser is created.  The function `generate_parser_with`
passes the given options to `Menhir` instead of `--lalr`, and it raises the exception `Grammar_error` if `Menhir`
rejects the grammar.  This corresponds to the keyword argument `strict=True` and the exception `GrammarError` of `Lark`.
It is a good idea to develop a grammar with `--strict`, for then a conflict cannot be overlooked.

In [ ]:
try generate_parser_with ~options:"--lalr --strict" ~name:"expr" ~lexer ~grammar
with Grammar_error e -> print_string e; flush stdout

## Inspecting the LALR States

Since `generate_parser` passes the options `--dump` and `--dump-resolved` to `Menhir`, the LALR automaton is written to
the file `_build/expr_parser.automaton`, and the automaton after the resolution of the conflicts is written to the file
`_build/expr_parser.automaton.resolved`.  The function `dump_states` prints the latter file.  It corresponds to the file
`parser.out` of `Ply`.  (The Python version has to dig the states out of the internal data structures of `Lark`.)

`Menhir` numbers its states, so there is no need to number them ourselves.  For every state, the file shows
- the *known stack suffix*, i.e. the symbols on top of the stack whenever the parser is in this state,
- the *LR(1) items*, i.e. the marked rules of the state together with their follow tokens in brackets.  `Menhir` only
  shows the marked rules of the *kernel* of the state, i.e. the marked rules that do not start with the marker.  The
  other marked rules of the state, which we have discussed in the lecture, can be computed from the kernel via the
  closure.
- the *transitions*, i.e. the shift actions for the tokens and the goto table for the variables, and
- the *reductions*.  A *default reduction* is performed without looking at the next token.

In [ ]:
let dump_states (name : string) : unit =
  print_string (read_file (Printf.sprintf "_build/%s_parser.automaton.resolved" name));
  flush stdout

Let us look at the action table that is generated.  Note that `Menhir` marks the position of the parser inside a rule
with the character `.` instead of the bullet `•` that we use in the lecture notes, that the end of the input is the
token `EOF`, and that the symbol `#` denotes the end of the token stream after `EOF`.

In the states that contain a marked rule of the form `expr -> expr PLUS expr .` or `expr -> expr STAR expr .` there are
two shift-reduce conflicts each.  All of them are resolved in favour of shifting: the only reduce action that survives
in these states is the one for `EOF`.  The discarded reduce actions are *not* shown here; in order to see them we have
to read the file `_build/expr_parser.automaton` or the explanations shown above.

In [ ]:
dump_states "expr"

## Building an Abstract Syntax Tree

The parse tree that is returned by the parser has the type `Parse_tree.tree`.  In Python, an abstract syntax tree is a
nested tuple like `('+', 1, 2)`.  In OCaml, we define the variant type `expr` instead.  In order to reuse the function
`tuple2dot`, the function `to_nested` converts an abstract syntax tree into a nested tuple of the type `nested` from
the notebook `AST2Dot.ipynb`.

The function `ast_builder` converts a parse tree into an abstract syntax tree.  It corresponds to the `Transformer` of
the Python version:  for every alias that occurs in the grammar, there is a case of the same name.  This case
converts the children and combines them into the value that replaces the node.

**Type safety:** The `Transformer` of `Lark` calls its methods by name, so a misspelled method is silently ignored and the
node remains in the tree.  In OCaml, `ast_builder` returns an `expr` in every case, so a parse tree that is not
converted completely raises an exception instead of producing a mixture of trees and tuples.

In [ ]:
run_notebook "../AST2Dot.ipynb"

In [ ]:
type expr =
  | Number of int
  | Add    of expr * expr
  | Mul    of expr * expr

let rec ast_builder (t : tree) : expr =
  match t with
  | Tree ("add", [ lhs; rhs ]) -> Add (ast_builder lhs, ast_builder rhs)
  | Tree ("mul", [ lhs; rhs ]) -> Mul (ast_builder lhs, ast_builder rhs)
  | Tree ("number", [ Token n ]) -> Number (int_of_string n.value)
  | _ -> invalid_arg ("ast_builder: unexpected node " ^ pretty t)

let rec to_nested (e : expr) : nested =
  match e with
  | Number n -> Int n
  | Add (lhs, rhs) -> Tuple ("+", [ to_nested lhs; to_nested rhs ])
  | Mul (lhs, rhs) -> Tuple ("*", [ to_nested lhs; to_nested rhs ])

The function `test s` takes a string `s` as its argument and tries to parse this string.  If all goes well, an abstract
syntax tree is displayed and returned as `Some t`.  If the string can't be parsed, the parser raises the exception
`Unexpected_input`, which we catch in order to print a readable error message.  In this case, `None` is returned.

In [ ]:
let test (s : string) : expr option =
  match parse_string Expr_parser.start Expr_lexer.token s with
  | exception Unexpected_input { message; line; column; position } ->
      Printf.printf "Syntax error:\n%s at line %d, column %d.\n\n%s%!" message line column (get_context s position);
      None
  | parse_tree ->
      let t = ast_builder parse_tree in
      let d = tuple2dot (to_nested t) in
      Graphviz.display d;
      Some t

The next example shows that this parser does not produce the abstract syntax that reflects the precedences
of the arithmetical operators:  the string `2*3+4` is parsed as `2*(3+4)`.

In [ ]:
test "2*3+4"

Due to the fact that all shift-reduce conflicts are resolved in favour of shift, all operators have the
same precedence and effectively associate to the right.

In [ ]:
test "2+3+4"

In [ ]:
test "1+2*3"

The notebook [02-Conflicts-Resolved.ipynb](02-Conflicts-Resolved.ipynb) shows how these conflicts can be removed.
`Menhir` supports operator precedence declarations, but the Python version uses `Lark`, which does not.  In order to
stay close to the Python version, the precedences and associativities of the operators are encoded into the grammar by
introducing one syntactical variable per precedence level.